# Topic 4 — AI Agent Patterns & Frameworks: Practice Notebook

Topics 1-3 gave you the primitives: LangChain's runnables and tools, LangGraph's
stateful graphs, and MCP's standardized way of exposing tools/resources/prompts to
any client. This notebook zooms out and looks at the **reasoning patterns** agents
use on top of those primitives, then tours how different frameworks package those
patterns up.

Sections:

1. ReAct Pattern — implemented by hand, no framework.
2. Plan-and-Execute — implemented by hand, no framework.
3. Reflection / Self-Critique — implemented by hand, no framework.
4. Framework Tour — the same "research a topic and write a 3-bullet summary with
   sources" task in LangGraph (provided), CrewAI (exercise), and the Claude Agent
   SDK (exercise).
5. Comparison & Decision Framework — a table you fill in based on section 4.
6. Agent-to-Agent (A2A) Protocol — a brief, illustrative look at how A2A
   complements MCP.

Sections 1-3 use `GenericFakeChatModel` with scripted responses, exactly like
Topics 1-3, so everything runs offline and deterministically. Section 4's
LangGraph reference reuses `mcp_server.py` (the same server from Topic 3, copied
into this folder) over stdio. Run this notebook from inside
`code/template/` (or `code/solutions/`) so the relative path `mcp_server.py`
resolves correctly.


In [ ]:
import re
import ast
import json
import operator
from typing import TypedDict, Annotated

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_mcp_adapters.client import MultiServerMCPClient

print("Imports OK. mcp_server.py (in this folder) is the MCP server from Topic 3, reused for Section 4.")


## 1. ReAct Pattern

**ReAct** ("Reasoning + Acting") interleaves a *Thought* (the model reasoning
about what to do next), an *Action* (a tool call with arguments), and an
*Observation* (the tool's result), repeating until the model emits a special
`Finish` action with the final answer.

In Topic 2, "tool calling" was hidden behind `bind_tools` and structured
`tool_calls` on an `AIMessage`. ReAct, in its original form, is older and more
primitive than that: the LLM is asked to produce **plain text** in a fixed
format, and the *agent code* parses that text to figure out which tool to call.
Seeing this raw form first makes it obvious what `bind_tools`/`ToolNode` are
automating for you.

Formally, with question $q$ and a tools registry $\{\text{Tool}_1, \dots,
\text{Tool}_k\}$, define the history $h_t = (T_1, A_1, O_1, \dots, T_{t-1},
A_{t-1}, O_{t-1})$. At step $t$:

$$T_t, A_t = \text{LLM}(q, h_t) \qquad O_t = \text{Tool}_{A_t.\text{name}}(A_t.\text{arg})$$

The loop stops the first time $A_t.\text{name} = \text{Finish}$, and the
answer is $A_t.\text{arg}$. $T_t$ (the "Thought") is never executed — it exists
purely so the model "thinks out loud" before committing to an action, which
empirically makes the chosen action more reliable.

We'll use a 2-tool problem: a `search` tool (a tiny in-memory knowledge base) and
a `calculator` tool (safe arithmetic), and ask a question that needs *both*:
"How many years passed between the introduction of the Transformer architecture
and the release of LangGraph?"


In [ ]:
KNOWLEDGE_BASE = {
    "transformer": "The Transformer architecture was introduced in 2017 in the paper 'Attention Is All You Need'.",
    "langgraph": "LangGraph was first released in 2024 as a library for building stateful, graph-based LLM applications.",
    "mcp": "The Model Context Protocol (MCP) was introduced by Anthropic in late 2024 as an open standard for connecting LLMs to tools and data.",
}


def search(query: str) -> str:
    """Look up a topic in the knowledge base."""
    return KNOWLEDGE_BASE.get(query.strip().lower(), "No results found.")


_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.USub: operator.neg,
}


def calculator(expression: str) -> str:
    """Safely evaluate a simple arithmetic expression like '2024 - 2017'."""

    def _eval(node):
        if isinstance(node, ast.Constant):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPERATORS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPERATORS[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsupported expression: {expression}")

    tree = ast.parse(expression, mode="eval")
    return str(_eval(tree.body))


TOOLS = {"search": search, "calculator": calculator}


def parse_action(text: str) -> tuple[str, str] | None:
    """Extract (action_name, action_argument) from a line like 'Action: search[langgraph]'.

    Returns None if the response contains no Action line.
    """
    match = re.search(r"Action:\s*(\w+)\[(.*)\]", text)
    if match is None:
        return None
    return match.group(1), match.group(2)


print("search('transformer') ->", search("transformer"))
print("calculator('2024 - 2017') ->", calculator("2024 - 2017"))
print("parse_action('Action: search[langgraph]') ->", parse_action("Action: search[langgraph]"))


In [ ]:
def run_react(llm, tools, question, max_steps=6):
    """Run a manual Thought/Action/Observation loop until a Finish action."""
    print(f"Question: {question}\n")
    scratchpad = ""
    for step in range(1, max_steps + 1):
        response = llm.invoke(f"Question: {question}\n{scratchpad}").content
        print(response)
        action = parse_action(response)
        if action is None:
            raise ValueError(f"No Action found in:\n{response}")
        action_name, action_arg = action
        if action_name == "Finish":
            print(f"\nFinal answer: {action_arg}")
            return action_arg
        observation = tools[action_name](action_arg)
        print(f"Observation: {observation}\n")
        scratchpad += f"{response}\nObservation: {observation}\n"
    raise RuntimeError("Max steps reached without a Finish action.")


In [ ]:
react_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="Thought: I need to find when the Transformer architecture was introduced.\nAction: search[transformer]"),
    AIMessage(content="Thought: Now I need to find when LangGraph was released.\nAction: search[langgraph]"),
    AIMessage(content="Thought: I have both years (2017 and 2024). I need to compute the difference.\nAction: calculator[2024 - 2017]"),
    AIMessage(content="Thought: I now know the final answer.\nAction: Finish[7 years passed between the introduction of the Transformer architecture (2017) and the release of LangGraph (2024).]"),
]))

run_react(
    react_llm,
    TOOLS,
    "How many years passed between the introduction of the Transformer architecture and the release of LangGraph?",
)


## 2. Plan-and-Execute

ReAct interleaves *one* reasoning step with *one* action, every time — every tool
call costs a full LLM round trip, and the model re-plans from scratch at every
step. **Plan-and-Execute** instead splits the work into two phases: a **planner**
LLM call produces an upfront list of steps, and an **executor** then works
through that list (calling tools where needed), with a final **synthesis** call
combining everything into the answer.

$$\text{Plan} = \pi_{\text{planner}}(q) = [s_1, s_2, \dots, s_n]$$

$$\text{For each } s_i: \quad A_i = \pi_{\text{executor}}(s_i), \qquad O_i =
\text{Tool}_{A_i.\text{name}}(A_i.\text{arg}) \text{ if } A_i \text{ is a tool call}$$

$$\text{Answer} = \pi_{\text{synth}}(q, O_1, \dots, O_k)$$

The tradeoff: Plan-and-Execute makes **fewer** LLM calls when the plan is good
(no per-step re-reasoning about *what to do next*, only *how to do this specific
step*), and the plan itself is a useful debugging artifact — you can read it
before any tool runs. But it's **less adaptive**: if step 2's result reveals that
step 3 no longer makes sense, a naive Plan-and-Execute loop (like the one below)
won't notice, whereas ReAct re-evaluates after every observation.

Task: "Write a one-paragraph summary comparing LangGraph and MCP, including the
year each was introduced." We reuse the `search` tool and its `KNOWLEDGE_BASE`
from Section 1 (which already has both `"langgraph"` and `"mcp"` entries).


In [ ]:
def parse_plan(text: str) -> list[str]:
    """Extract numbered steps ('1. ...', '2. ...') from a planner response."""
    return re.findall(r"^\d+\.\s*(.+)$", text, flags=re.MULTILINE)


def run_plan_and_execute(planner_llm, executor_llm, synthesis_llm, tools, task):
    print(f"Task: {task}\n")

    plan_response = planner_llm.invoke(f"Task: {task}\nProduce a numbered plan.").content
    print(plan_response)
    steps = parse_plan(plan_response)

    observations = []
    for i, step in enumerate(steps, start=1):
        print(f"\nExecuting step {i}: {step}")
        step_response = executor_llm.invoke(f"Step: {step}").content
        action = parse_action(step_response)
        if action is None:
            print(f"  {step_response}")
            break
        action_name, action_arg = action
        observation = tools[action_name](action_arg)
        print(f"  {step_response}")
        print(f"  Observation: {observation}")
        observations.append(observation)

    context = "\n".join(observations)
    final_answer = synthesis_llm.invoke(f"Task: {task}\nFindings:\n{context}\nWrite the final answer.").content
    print(f"\nFinal answer:\n{final_answer}")
    return final_answer


In [ ]:
planner_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content=(
        "Plan:\n"
        "1. Look up information about LangGraph.\n"
        "2. Look up information about MCP.\n"
        "3. Combine both lookups into a one-paragraph summary mentioning both release years."
    )),
]))

executor_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="Action: search[langgraph]"),
    AIMessage(content="Action: search[mcp]"),
    AIMessage(content="No tool needed -- ready to write the summary."),
]))

synthesis_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content=(
        "LangGraph and MCP were both introduced in 2024: LangGraph is a library for building "
        "stateful, graph-based LLM applications, while MCP is an open standard for connecting "
        "LLMs to tools and data. Together they form a complementary stack -- LangGraph "
        "orchestrates an agent's control flow, and MCP standardizes how that agent reaches "
        "external tools."
    )),
]))

run_plan_and_execute(
    planner_llm,
    executor_llm,
    synthesis_llm,
    {"search": search},
    "Write a one-paragraph summary comparing LangGraph and MCP, including the year each was introduced.",
)


## 3. Reflection / Self-Critique

Both ReAct and Plan-and-Execute stop as soon as the model produces *an* answer —
neither pattern checks whether that answer is any *good*. **Reflection** adds one
more role, a **critic**, which reviews the draft and either approves it
(`Verdict: PASS`) or asks for a revision (`Verdict: REVISE`, with specific
feedback). The writer then revises using that feedback, and the cycle repeats up
to some maximum number of rounds.

$$d_1 = \pi_{\text{writer}}(q)$$

$$\text{For round } r = 1, 2, \dots: \quad c_r = \pi_{\text{critic}}(q, d_r)$$

$$\text{if } c_r = \texttt{PASS}: \text{ return } d_r \qquad \text{else: } d_{r+1} = \pi_{\text{writer}}(q, d_r, c_r)$$

This costs at least one extra LLM call (the critique) per round, and potentially
a second extra call (the revision) — for a 2-round reflection, that's up to 4x
the LLM calls of a single draft. It's worth it when correctness or quality is
hard to get right in one shot but easy to *check* — e.g. "does this summary
actually explain the mechanism, not just name it?" — which is exactly the kind
of thing a second LLM call (or even a rubric-based check) can catch reliably.

Task: "Write a 2-sentence summary of what self-attention does."


In [ ]:
def run_reflection(writer_llm, critic_llm, task, max_rounds=2):
    print(f"Task: {task}\n")

    draft = writer_llm.invoke(f"Task: {task}\nWrite a draft.").content
    print(f"Draft 1:\n{draft}\n")

    for round_num in range(1, max_rounds + 1):
        critique = critic_llm.invoke(f"Task: {task}\nDraft:\n{draft}\nCritique this draft.").content
        print(f"Critique (round {round_num}):\n{critique}\n")

        if critique.startswith("Verdict: PASS"):
            print("Critic approved -- stopping.")
            return draft

        draft = writer_llm.invoke(f"Task: {task}\nDraft:\n{draft}\nCritique:\n{critique}\nRevise the draft.").content
        print(f"Draft {round_num + 1}:\n{draft}\n")

    print("Max rounds reached -- returning latest draft.")
    return draft


In [ ]:
writer_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="Self-attention is a mechanism used in transformers."),
    AIMessage(content=(
        "Self-attention lets each token in a sequence weigh and combine information from every "
        "other token, producing context-aware representations. This allows transformers to "
        "capture long-range dependencies without recurrence."
    )),
]))

critic_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content=(
        "Verdict: REVISE. The draft says self-attention is 'used in transformers' but doesn't "
        "explain what it actually computes or why it's useful -- add the token-to-token "
        "weighting mechanism and its benefit."
    )),
    AIMessage(content="Verdict: PASS. The revised draft explains both the mechanism and its benefit."),
]))

run_reflection(
    writer_llm,
    critic_llm,
    "Write a 2-sentence summary of what self-attention does.",
)


## 4. Framework Tour

Sections 1-3 hand-rolled three control-flow patterns. Real frameworks package
these patterns (and the tool-calling/state-management plumbing underneath them)
behind higher-level APIs. To compare them apples-to-apples, all three
implementations below tackle the **same task**:

> Research the topic **"attention"** using the `learning-notes` MCP server (from
> Topic 3, `mcp_server.py` in this folder) and write a **3-bullet summary with
> sources**.

### LangGraph (reference)

This is the agent/tools graph from Topic 3 Section 4, renamed for this task. It's
provided as a *reference* — you've already built and understood this pattern.


In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


def build_research_agent_graph(tools, llm):
    def agent_node(state: AgentState) -> dict:
        return {"messages": [llm.invoke(state["messages"])]}

    graph = StateGraph(AgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", tools_condition)
    graph.add_edge("tools", "agent")
    return graph.compile()


In [ ]:
client = MultiServerMCPClient({
    "learning_notes": {
        "transport": "stdio",
        "command": "python3",
        "args": ["mcp_server.py"],
    }
})
mcp_tools = await client.get_tools()
print("Tools loaded:", [t.name for t in mcp_tools])

agent_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="", tool_calls=[{"name": "search_notes", "args": {"query": "attention"}, "id": "call_1"}]),
    AIMessage(content=(
        "Here is a 3-bullet summary of self-attention:\n"
        "- Self-attention lets each token weigh and combine information from every other token.\n"
        "- It produces context-aware representations without recurrence.\n"
        "- Source: learning-notes database, topic 'attention'."
    )),
]))

research_app = build_research_agent_graph(mcp_tools, agent_llm)
result = await research_app.ainvoke({"messages": [HumanMessage(content="Research self-attention and write a 3-bullet summary with sources.")]})
for message in result["messages"]:
    print(type(message).__name__, "|", repr(message.content))


### Exercise 1 — CrewAI version

[CrewAI](https://docs.crewai.com/) models work as a **crew** of role-based
agents, each with a `role`, `goal`, and `backstory`, assigned `Task`s that a
`Crew` runs (by default, sequentially — each task's output feeds into the next).

This exercise requires `pip install crewai` and an LLM provider API key (CrewAI
uses [LiteLLM](https://docs.litellm.ai/) under the hood, so `OPENAI_API_KEY`,
`ANTHROPIC_API_KEY`, etc. all work) — it is **not** executed in this notebook,
which must stay offline/deterministic. The cell below checks whether `crewai` is
installed and prints a message if not; if it *is* installed (and you have an API
key configured), fill in the `# YOUR CODE HERE` section and re-run the cell.

1. Define a `researcher` `Agent` (`role="Researcher"`, a `goal` about looking up
   topics and a short `backstory`).
2. Define a `writer` `Agent` (`role="Writer"`, a `goal` about turning research
   into a 3-bullet summary with sources).
3. Define a research `Task` assigned to `researcher` — `description` should ask
   it to research "attention" (you can give it the `KNOWLEDGE_BASE`/`search` tool
   from Section 1, or describe the fact directly in the task).
4. Define a writing `Task` assigned to `writer`, with `context=[research_task]`
   so it receives the research task's output, asking for a 3-bullet summary with
   sources.
5. Build `Crew(agents=[researcher, writer], tasks=[research_task, writing_task],
   process=Process.sequential)` and call `crew.kickoff()`.
6. `print(result)`.


In [ ]:
try:
    from crewai import Agent, Task, Crew, Process
    CREWAI_AVAILABLE = True
except ImportError:
    CREWAI_AVAILABLE = False

print(f"crewai available: {CREWAI_AVAILABLE}")

if not CREWAI_AVAILABLE:
    print("Skipping -- install with `pip install crewai` and set an LLM provider API key "
          "(e.g. OPENAI_API_KEY or ANTHROPIC_API_KEY) to complete this exercise.")
else:
    # YOUR CODE HERE
    pass


### Exercise 2 — Claude Agent SDK version

The [Claude Agent SDK](https://docs.claude.com/en/api/agent-sdk/overview) is the
same engine that powers Claude Code: you send a prompt (and optional
`ClaudeAgentOptions`, e.g. a `system_prompt` and allowed tools/MCP servers), and
it streams back messages as the agent reasons, calls tools, and responds — the
agent loop itself is entirely managed by the SDK.

This exercise requires `pip install claude-agent-sdk`, the Claude Code CLI
available on `PATH`, and `ANTHROPIC_API_KEY` set — it is **not** executed in this
notebook. The cell below checks whether `claude_agent_sdk` is installed and
prints a message if not; if it *is* available, fill in the
`# YOUR CODE HERE` section and re-run the cell.

1. Build `options = ClaudeAgentOptions(system_prompt="...")`, instructing the
   agent to research a topic and respond with a 3-bullet summary citing sources.
2. `async for message in query(prompt="Research self-attention and write a "
   "3-bullet summary with sources.", options=options):`
3. Inside the loop, `print(message)` to see the streamed agent trace.


In [ ]:
try:
    from claude_agent_sdk import query, ClaudeAgentOptions
    CLAUDE_SDK_AVAILABLE = True
except ImportError:
    CLAUDE_SDK_AVAILABLE = False

print(f"claude_agent_sdk available: {CLAUDE_SDK_AVAILABLE}")

if not CLAUDE_SDK_AVAILABLE:
    print("Skipping -- install with `pip install claude-agent-sdk`, ensure the Claude Code CLI "
          "is on PATH, and set ANTHROPIC_API_KEY to complete this exercise.")
else:
    # YOUR CODE HERE
    pass


## 5. Comparison & Decision Framework

Fill in the **CrewAI** and **Claude Agent SDK** rows yourself once you've worked
through the exercises in Section 4 (with an API key configured). The
**LangGraph** row is filled in based on what you've built across Topics 2-4.

| Framework | Lines of code (approx.) | Debuggability | Persistence support | Multi-agent support | MCP / A2A support |
|---|---|---|---|---|---|
| **LangGraph** | ~10 (graph definition) + reused MCP tool loading | High — every node/edge is explicit Python; `result["messages"]` shows the full trace, including raw `ToolMessage` content | Built-in via checkpointers (Topic 2) — pause/resume any thread | Possible, but manual: each agent is its own node or subgraph you wire up yourself | Native — `langchain-mcp-adapters` (Topic 3); A2A via community adapters |
| **CrewAI** | _fill in_ | _fill in_ | _fill in_ | _fill in_ | _fill in_ |
| **Claude Agent SDK** | _fill in_ | _fill in_ | _fill in_ | _fill in_ | _fill in_ |

Some questions to guide your fill-in: How many lines did *you* have to write vs.
how much did the framework provide for free? When something went wrong, could you
see *why* (intermediate messages/state), or only the final output? Could you
stop and resume a run? How natural was it to add a second agent/role? Did the
framework have first-class MCP support, or did you have to bridge it yourself?


## 6. Agent-to-Agent (A2A) Protocol

MCP (Topic 3) standardizes how an agent talks to **tools, resources, and
prompts**. **A2A** (Agent-to-Agent), an open protocol originally from Google and
now under the Linux Foundation, standardizes something different: how one
**agent** discovers and talks to **another agent** — potentially built by a
different team, in a different framework, or by a different company entirely.

The two are complementary, not competing: an agent might use MCP to call a
`search_notes` *tool*, and A2A to delegate a sub-task to a separate "research
agent" *service* that itself might be built in CrewAI, LangGraph, or anything
else. The "agent" on the other side of an A2A call could easily be exposing its
own internal tools via MCP — A2A doesn't replace MCP, it sits one level up.

The core A2A primitive is the **Agent Card**: a JSON document (typically served
at `/.well-known/agent.json`) that describes an agent's identity, capabilities,
and **skills** — conceptually similar to MCP's `tools/list` response, but
describing an entire agent's *services* rather than individual *functions*. A
client agent fetches another agent's Agent Card to discover what it can do
*before* sending it any work.


In [ ]:
agent_card = {
    "name": "learning-notes-research-agent",
    "description": "Researches topics in the learning notes database and writes a 3-bullet summary with sources.",
    "url": "https://example.local/a2a/research-agent",
    "version": "1.0.0",
    "capabilities": {"streaming": True, "pushNotifications": False},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": [
        {
            "id": "research-and-summarize",
            "name": "Research and summarize",
            "description": "Look up a topic in the learning notes database and produce a 3-bullet summary with sources.",
            "tags": ["research", "summarization"],
        }
    ],
}

print(json.dumps(agent_card, indent=2))


## Putting It All Together

```
ReAct (Section 1)                Plan-and-Execute (Section 2)         Reflection (Section 3)

  Question                          Question                            Question
     |                                 |                                   |
     v                                 v                                   v
 +---------+                     +-----------+                       +--------+
 | Thought |<-----+              |  Planner  |                       | Writer |
 +---------+      |              +-----------+                       +--------+
     |            |                    |                                  |
     v            |                    v                                  v
 +--------+       |              [s1, s2, ..., sn]                     Draft 1
 | Action |       |                    |                                  |
 +--------+       |                    v                                  v
     |            |              +-----------+                      +--------+
     v            |              | Executor  |--(tool calls)-->     | Critic |
 +-------------+  |              +-----------+                      +--------+
 | Observation |--+                    |                                  |
 +-------------+                       v                          PASS /  | REVISE
     |                            +-----------+                    |      v
     v (Finish)                   | Synthesis |                    |  Draft r+1 -> Critic
  Answer                          +-----------+                    v
                                        |                        Answer
                                        v
                                     Answer
```

Section 4's three implementations are all just *different ways to wire up* one
of these patterns (LangGraph and CrewAI both default to something close to ReAct
under the hood; the Claude Agent SDK's agent loop is also fundamentally a ReAct
variant). Knowing the hand-rolled version means that when a framework's agent
"gets stuck" or does something unexpected, you have a mental model of *what loop
is running* and *where to look* — the framework is doing Sections 1-3 for you,
not something fundamentally different.


## Where to Go Next

Topic 5 (Advanced RAG) moves from *agent control flow* to *retrieval quality* —
hybrid search, reranking, and agentic/GraphRAG patterns where an agent (built
with exactly the patterns from this notebook) decides *when* and *how* to
retrieve, rather than retrieving once at the start of every query.
